Manually creating multiple partition

In [1]:
from pyspark.sql import SparkSession

# Start Spark
spark = SparkSession.builder.master("local[*]").appName("PartitionData").getOrCreate()
sc = spark.sparkContext

# Create RDD with 10 items and 3 partitions
rdd = sc.parallelize(range(50), 5)

# Show data in each partition
def show_partitions(rdd):
    def inspect(index, iterator):
        yield f"Partition {index}: {list(iterator)}"
    result = rdd.mapPartitionsWithIndex(inspect).collect()
    for item in result:
        print(item)

# Run it
show_partitions(rdd)

Partition 0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Partition 1: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Partition 2: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Partition 3: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
Partition 4: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


Manual Partions with external file data

In [8]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession\
        .builder\
        .master('local[*]')\
        .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

from google.colab import drive
drive.mount('/content/drive')


# Load CSV into RDD with more partitions (e.g. 4)
rdd = sc.textFile('/content/drive/My Drive/Colab Notebooks/sales.txt', minPartitions=4)
print("Partitions:", rdd.getNumPartitions())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Partitions: 4


In [9]:
# Check number of partitions
print("Partitions:", rdd.getNumPartitions())



Partitions: 4


Display partitionwise data

In [10]:
def show_partitions(rdd):
    def inspect(index, iterator):
        yield f"Partition {index}: {list(iterator)}"
    for item in rdd.mapPartitionsWithIndex(inspect).collect():
        print(item)

show_partitions(rdd)

Partition 0: ['date,region,sales', '2024-01-01,North,100', '2024-01-01,South,150', '2024-01-02,North,200', '2024-01-02,West,300', '2024-01-03,South,120', '2024-01-03,East,180', '2024-01-04,North,100', '2024-01-04,South,150']
Partition 1: ['2024-01-04,North,200', '2024-01-05,West,300', '2024-01-05,South,120', '2024-01-05,East,180', '2024-01-06,East,180', '2024-01-06,North,100', '2024-01-06,South,150', '2024-01-07,North,200']
Partition 2: ['2024-01-07,West,300', '2024-01-07,South,120', '2024-01-08,North,100', '2024-01-08,South,150', '2024-01-08,West,300', '2024-01-08,East,180', '2024-01-08,North,100']
Partition 3: ['2024-01-09,South,150', '2024-01-09,North,200', '2024-01-09,West,300', '2024-01-09,East,180', '2024-01-10,East,180', '2024-01-10,North,100', '2024-01-10,South,150', '2024-01-10,West,300']


In [13]:
# Transform to (region, sales) pairs
region_sales = (
    rdd.map(lambda line: line.split(","))       # Split CSV rows
                 .filter(lambda fields: fields[0] != 'date') # Filter out header row
                 .map(lambda fields: (fields[1], int(fields[2])))  # (region, sales)
)

# Reduce by region
total_sales = region_sales.reduceByKey(lambda a, b: a + b)

# Collect and print results
for region, total in total_sales.collect():
    print(f"{region}: {total}")

West: 1800
North: 1400
South: 1260
East: 1080
